In [ ]:
import json
from sklearn.metrics import classification_report
import pandas as pd
import sys, os
sys.path.insert(0, "/Users/abhinavarora/Desktop/CadenceCV/ml/src")
from training.data_loader import training_df, testing_df

In [ ]:
random_forest_metrics = ["ankle_approach_angle", "knee_flexion_angle", "tibial_angle", 
            "trunk_angle", "ankle_hip_horizontal_offset", "ankle_x_vel", 
            "ankle_y_vel", "shin_velocity", "foot_angle", "strike_pattern"]

def obtain_data(dataframe:pd.DataFrame):
    df = dataframe[random_forest_metrics]
    #Training data
    df.loc[(df.strike_pattern == "midfoot") | (df.strike_pattern == "forefoot"), "strike_pattern"] = "m/f"
    #Randomly shuffling the data
    df = df.sample(frac=1).reset_index(drop=True)
    df_labels = pd.DataFrame(df["strike_pattern"])

    # Heel strikes should have positive foot angle (toe above heel)
    # m/f strikes should have negative or near-zero foot angle
    suspicious = df[(df["strike_pattern"] == "heel") & (df["foot_angle"] < -20)] 
    suspicious2 = df[(df["strike_pattern"] == "m/f") & (df["foot_angle"] > 20)]

    print(len(suspicious), "suspicious heel labels")
    print(len(suspicious2), "suspicious m/f labels")
    #Removing the strike_pattern column
    df = df.drop("strike_pattern", axis=1)

    print(df_labels.value_counts())
    # Dropping rows with missing foot_angle values
    missing = df["foot_angle"].isna()
    # ~ is the bitwise NOT operator
    df = df[~missing]
    df_labels = df_labels[~missing]

    return df, df_labels

#Obtaining the correct training and testing data
training_df_rf, training_df_labels = obtain_data(training_df)
testing_df_rf, testing_df_labels = obtain_data(testing_df)